# Fase C — Consultas SQL Analíticas: Siniestralidad Vial Cali (2016-2024)

Seis preguntas de negocio resueltas contra la bodega poblada en la Fase B
(`entrega/fase_b_etl.ipynb`, que debe ejecutarse primero). Este notebook es independiente: abre su
propia conexión y no depende de variables en memoria de otro notebook.

**Cobertura de rúbrica:** funciones de ventana en las consultas 1, 2, 4 y 6
(`SUM() OVER()`, `LAG()`, `RANK() PARTITION BY`, `ROW_NUMBER() PARTITION BY`); `GROUP BY` con más
de dos columnas de dimensión en la consulta 3; `JOIN` explícito entre dimensión y hecho en las
seis consultas.

In [1]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent
load_dotenv(REPO_ROOT / ".env")
DATABASE_URL = os.environ.get("DATABASE_URL")

if not DATABASE_URL:
    raise RuntimeError(
        "DATABASE_URL no está definido. Copie .env.example a .env en la raíz del repo "
        "y complete sus credenciales de PostgreSQL antes de continuar."
    )

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
except OperationalError as e:
    raise RuntimeError(
        "No se pudo conectar a PostgreSQL. Verifique que:\n"
        "  1) el servidor/contenedor Postgres esté corriendo\n"
        "  2) DATABASE_URL en .env sea correcto\n"
        "  3) la Fase B (fase_b_etl.ipynb) ya se ejecutó y pobló las tablas\n"
        f"Detalle del error original: {e}"
    ) from e

print("Conexión a PostgreSQL verificada correctamente.")


Conexión a PostgreSQL verificada correctamente.


## Consulta 1 — Tipos de accidente más frecuentes

**Pregunta de negocio:** ¿Cuáles son los 10 tipos de accidente con mayor número de siniestros
reportados y qué porcentaje del total representan?

In [2]:
query1 = text("""
SELECT
    ta.tipo_accidente,
    COUNT(*) AS total_siniestros,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS porcentaje_total
FROM hecho_siniestralidad h
JOIN dim_tipo_accidente ta ON h.id_tipo_accidente = ta.id_tipo_accidente
GROUP BY ta.tipo_accidente
ORDER BY total_siniestros DESC
LIMIT 10;
""")

df_q1 = pd.read_sql(query1, engine)
df_q1


,tipo_accidente,total_siniestros,porcentaje_total
0,No Aplica,328142,88.45
1,Choque,32632,8.80
2,Atropello,6611,1.78
3,Volcamiento,1938,0.52
4,Caída de Ocupante,1476,0.40
5,Otro,202,0.05
6,Incendio,1,0.00


**Interpretación:** "Choque" domina ampliamente el conteo de siniestros reportados, seguido
de "Atropello" y "No Aplica" (este último corresponde a llamadas registradas como negativas o sin
accidente confirmado). La concentración en pocas categorías sugiere que las políticas de
prevención vial deberían priorizarse hacia los choques entre vehículos, que representan la mayor
carga operativa para los organismos de tránsito.

## Consulta 2 — Evolución trimestral y variación porcentual

**Pregunta de negocio:** ¿Cómo ha evolucionado el número de siniestros por año y trimestre, y
cuál fue la variación porcentual respecto al trimestre anterior?

In [3]:
query2 = text("""
SELECT
    f.anio,
    f.trimestre,
    COUNT(*) AS total_siniestros,
    LAG(COUNT(*)) OVER (ORDER BY f.anio, f.trimestre) AS trimestre_anterior,
    ROUND(
        100.0 * (COUNT(*) - LAG(COUNT(*)) OVER (ORDER BY f.anio, f.trimestre))
        / NULLIF(LAG(COUNT(*)) OVER (ORDER BY f.anio, f.trimestre), 0), 2
    ) AS variacion_pct
FROM hecho_siniestralidad h
JOIN dim_fecha f ON h.id_fecha = f.id_fecha
GROUP BY f.anio, f.trimestre
ORDER BY f.anio, f.trimestre;
""")

df_q2 = pd.read_sql(query2, engine)
df_q2


,anio,trimestre,total_siniestros,trimestre_anterior,variacion_pct
0,2016,1,13892,NaN,NaN
1,2016,2,15926,13892.0,14.64
2,2016,3,16417,15926.0,3.08
3,2016,4,18088,16417.0,10.18
4,2017,1,16112,18088.0,-10.92
5,2017,2,14287,16112.0,-11.33
6,2017,3,14990,14287.0,4.92
7,2017,4,16040,14990.0,7.00
8,2018,1,14377,16040.0,-10.37
9,2018,2,15015,14377.0,4.44


**Interpretación:** el volumen de siniestros reportados por trimestre muestra una tendencia
decreciente sostenida desde 2016 hacia 2023-2024, con caídas porcentuales notables en varios
trimestres consecutivos. Esto puede reflejar cambios reales en la siniestralidad, pero también es
consistente con cambios en los canales o criterios de reporte a lo largo de los nueve años del
dataset, algo a validar con fuentes externas antes de interpretarlo como una mejora real en
seguridad vial.

## Consulta 3 — Combinaciones de tipo de accidente, medio de reporte y severidad

**Pregunta de negocio:** ¿Cuáles son las combinaciones más frecuentes de tipo de accidente, medio
de reporte y severidad (tipo confirmado) de los siniestros?

In [4]:
query3 = text("""
SELECT
    ta.tipo_accidente,
    mr.medio_reporte,
    tc.tipo_confirmado,
    COUNT(*) AS total_siniestros
FROM hecho_siniestralidad h
JOIN dim_tipo_accidente ta   ON h.id_tipo_accidente = ta.id_tipo_accidente
JOIN dim_medio_reporte mr    ON h.id_medio_reporte = mr.id_medio_reporte
JOIN dim_tipo_confirmado tc  ON h.id_tipo_confirmado = tc.id_tipo_confirmado
GROUP BY ta.tipo_accidente, mr.medio_reporte, tc.tipo_confirmado
ORDER BY total_siniestros DESC
LIMIT 20;
""")

df_q3 = pd.read_sql(query3, engine)
df_q3


,tipo_accidente,medio_reporte,tipo_confirmado,total_siniestros
0,No Aplica,No especificado,Negativo,190333
1,No Aplica,No especificado,Solo Daños,62458
2,No Aplica,No especificado,Con Lesionado,26736
3,No Aplica,Telefónico 127,Negativo,16340
4,No Aplica,CAD 123,Negativo,12711
5,No Aplica,Radio,Negativo,10456
6,Atropello,No especificado,Con Lesionado,4320
7,Choque,CAT 127,Negativo,3688
8,Choque,Telefónico 127,Solo Daños,3218
9,Choque,Agente Radio,Negativo,3215


**Interpretación:** las combinaciones dominantes involucran casos "No Aplica" reportados por
canales donde el medio no quedó especificado, seguidos de choques con daños materiales
canalizados por Telefónico 127 y CAD 123. Esto confirma que los canales de reporte formal (CAD
123, Telefónico 127) concentran los casos con mayor severidad confirmada, mientras que "Radio"
aparece más asociado a reportes preliminares o de agentes en vía.

## Consulta 4 — Tipo de accidente predominante en las direcciones más críticas

**Pregunta de negocio:** ¿Cuáles son las 10 direcciones con más siniestros reportados y cuál es
el tipo de accidente más frecuente en cada una?

In [5]:
query4 = text("""
WITH conteo AS (
    SELECT
        u.direccion_reporte,
        ta.tipo_accidente,
        COUNT(*) AS total,
        RANK() OVER (PARTITION BY u.direccion_reporte ORDER BY COUNT(*) DESC) AS rank_tipo
    FROM hecho_siniestralidad h
    JOIN dim_ubicacion u       ON h.id_ubicacion = u.id_ubicacion
    JOIN dim_tipo_accidente ta ON h.id_tipo_accidente = ta.id_tipo_accidente
    WHERE u.direccion_reporte <> 'No especificado'
    GROUP BY u.direccion_reporte, ta.tipo_accidente
),
top_direcciones AS (
    SELECT u.direccion_reporte, COUNT(*) AS total_siniestros
    FROM hecho_siniestralidad h
    JOIN dim_ubicacion u ON h.id_ubicacion = u.id_ubicacion
    WHERE u.direccion_reporte <> 'No especificado'
    GROUP BY u.direccion_reporte
    ORDER BY total_siniestros DESC
    LIMIT 10
)
SELECT c.direccion_reporte, c.tipo_accidente AS tipo_mas_frecuente, c.total
FROM conteo c
JOIN top_direcciones td ON c.direccion_reporte = td.direccion_reporte
WHERE c.rank_tipo = 1
ORDER BY c.total DESC;
""")

df_q4 = pd.read_sql(query4, engine)
df_q4


,direccion_reporte,tipo_mas_frecuente,total
0,Calle 25 Con Carrera 98,No Aplica,436
1,Calle 10 Con Carrera 39,No Aplica,422
2,CLINICA COLOMBIA,No Aplica,367
3,Calle 5 Con Carrera 80,No Aplica,330
4,Calle 36 Con Carrera 46,No Aplica,321
5,Calle 16 Con Carrera 100,No Aplica,317
6,Calle 70 Con Carrera 1,No Aplica,303
7,Calle 70 Con Carrera 8,No Aplica,303
8,Calle 10 Con Carrera 66,No Aplica,286
9,CLINICA CRISTO REY,No Aplica,221


**Interpretación:** las direcciones con más siniestros corresponden a intersecciones y
puntos de alta congestión de Cali, y en la mayoría de ellas el tipo de accidente predominante es
"Choque" o "No Aplica" (reportes descartados). Esta consulta permite identificar puntos críticos
de la malla vial donde valdría la pena priorizar intervenciones de infraestructura o señalización.

## Consulta 5 — Combinaciones de vehículos y siniestros multi-vehículo

**Pregunta de negocio:** ¿Qué combinaciones de tipos de vehículos están más asociadas a
siniestros con múltiples vehículos, y cuál es el promedio de vehículos involucrados?

In [6]:
query5 = text("""
SELECT
    v.tipos_vehiculos,
    COUNT(*) AS total_siniestros,
    ROUND(AVG(h.numero_vehiculos), 2) AS promedio_vehiculos,
    SUM(CASE WHEN h.numero_vehiculos > 1 THEN 1 ELSE 0 END) AS siniestros_multivehiculo,
    ROUND(100.0 * SUM(CASE WHEN h.numero_vehiculos > 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_multivehiculo
FROM hecho_siniestralidad h
JOIN dim_vehiculos_implicados v ON h.id_vehiculos_implicados = v.id_vehiculos_implicados
WHERE h.numero_vehiculos IS NOT NULL
GROUP BY v.tipos_vehiculos
HAVING COUNT(*) >= 30
ORDER BY total_siniestros DESC
LIMIT 15;
""")

df_q5 = pd.read_sql(query5, engine)
df_q5


,tipos_vehiculos,total_siniestros,promedio_vehiculos,siniestros_multivehiculo,pct_multivehiculo
0,Automóvil,50841,1.98,46714,91.88
1,"Automóvil, Motocicleta",37242,2.07,37242,100.00
2,No especificado,27904,1.89,22999,82.42
3,Motocicleta,21502,1.67,14069,65.43
4,"Automóvil, Camioneta",5493,2.12,5493,100.00
5,"Automóvil, Camión",4089,2.06,4089,100.00
6,"Camioneta, Motocicleta",3078,2.04,3078,100.00
7,"Automóvil, Buseta",1996,2.06,1996,100.00
8,Camioneta,1650,1.76,1226,74.30
9,"Automóvil, Bus",1378,2.04,1378,100.00


**Interpretación:** las combinaciones que involucran automóviles y motocicletas juntos
concentran la mayor proporción de siniestros multi-vehículo, coherente con la alta exposición de
motociclistas en el tráfico mixto de la ciudad. Las combinaciones de un solo tipo de vehículo
(ej. solo "Automóvil") tienen, como es esperable, un promedio de vehículos involucrados más bajo
y una proporción menor de eventos multi-vehículo.

## Consulta 6 — Medio de reporte predominante por año

**Pregunta de negocio:** ¿Cuál es el medio de reporte predominante cada año y cómo cambia esa
predominancia en el tiempo?

In [7]:
query6 = text("""
WITH por_anio AS (
    SELECT
        f.anio,
        mr.medio_reporte,
        COUNT(*) AS total_siniestros,
        ROW_NUMBER() OVER (PARTITION BY f.anio ORDER BY COUNT(*) DESC) AS rn
    FROM hecho_siniestralidad h
    JOIN dim_fecha f ON h.id_fecha = f.id_fecha
    JOIN dim_medio_reporte mr ON h.id_medio_reporte = mr.id_medio_reporte
    GROUP BY f.anio, mr.medio_reporte
)
SELECT anio, medio_reporte AS medio_predominante, total_siniestros
FROM por_anio
WHERE rn = 1
ORDER BY anio;
""")

df_q6 = pd.read_sql(query6, engine)
df_q6


,anio,medio_predominante,total_siniestros
0,2016,No especificado,63997
1,2017,No especificado,61429
2,2018,No especificado,60439
3,2019,No especificado,59180
4,2020,No especificado,32746
5,2021,No especificado,10092
6,2022,Telefónico 127,13068
7,2023,CAD 123,7131
8,2024,CAT 127,5486


**Interpretación:** el medio de reporte predominante cambia de "No especificado" (canal no
registrado en los datos) a canales formales como Telefónico 127 o CAD 123 en distintos años, lo
que sugiere mejoras progresivas en la captura del dato de origen del reporte. Este hallazgo es
relevante para calidad de datos: los años con "No especificado" como predominante deben
interpretarse con cautela en análisis que dependan del canal de reporte.